# Candidate Recommendation (Employer) — Embedding Improved (Rerank + Proxy Evaluation)

**Goal:** Recommend top candidates for a given job posting using:
1) **Semantic retrieval** (Sentence-BERT)
2) **Reranking** with:
   - taxonomy match
   - seniority alignment
   - skill overlap
3) **Proxy evaluation** (no ground-truth job↔candidate labels in dataset)

**Outputs**
- `./outputs/candrec_emb/improved_topk.csv`
- `./outputs/candrec_emb/metrics_rows.csv`
- `./outputs/candrec_emb/metrics_summary.csv`


In [13]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("./datasets")
OUT_DIR  = Path("./outputs/candrec_emb")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ✅ ĐỪNG dùng DATA_DIR / "/Users/..." (sai). Dùng path tuyệt đối thẳng luôn:
JOBS_XLSX = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx")
CVS_CSV   = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")

jobs = pd.read_excel(JOBS_XLSX)
cvs  = pd.read_csv(CVS_CSV)

def norm_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ✅ JOBS: tự chọn đúng cột title + description có thật
TITLE_COL = None
for c in ["job_title", "title", "Job Title"]:
    if c in jobs.columns:
        TITLE_COL = c
        break

DESC_COL = None
for c in ["clean_jd", "job_description", "description", "jd", "Job Description", "cleaned_description"]:
    if c in jobs.columns:
        DESC_COL = c
        break

if TITLE_COL is None:
    raise ValueError(f"Không tìm thấy cột title trong jobs. Columns: {jobs.columns.tolist()}")

if DESC_COL is None:
    # nếu không có mô tả thì vẫn chạy được nhưng job_text sẽ nghèo
    jobs["__desc__"] = ""
    DESC_COL = "__desc__"

jobs[TITLE_COL] = jobs[TITLE_COL].fillna("").map(norm_text)
jobs[DESC_COL]  = jobs[DESC_COL].fillna("").map(norm_text)

jobs["job_text"] = (jobs[TITLE_COL] + ". " + jobs[DESC_COL]).map(norm_text)

# ✅ CVS: dùng đúng cột Resume
cvs["Category"] = cvs["Category"].fillna("").map(norm_text)
cvs["Resume"]   = cvs["Resume"].fillna("").map(norm_text)
cvs["clean_resume"] = cvs["Resume"]  # đặt alias cho thống nhất pipeline

print("jobs:", jobs.shape, "| cvs:", cvs.shape)
print("TITLE_COL =", TITLE_COL, "| DESC_COL =", DESC_COL)
print("job_text sample:", jobs.loc[0, "job_text"][:300])


jobs: (89277, 6) | cvs: (962, 3)
TITLE_COL = title | DESC_COL = description
job_text sample: Support Technologist II 2024-02216. **description and functions** ----------------------------- **open until filled** **general description:** this position is an entry-level position that, with guidance, provides technical support, technical expertise, customer satisfaction, and timeliness to assig


In [14]:

from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

# cache resume embeddings
emb_path = OUT_DIR / f"resume_embeddings_{MODEL_NAME.split('/')[-1]}.npy"
if emb_path.exists():
    resume_emb = np.load(emb_path)
    print("Loaded cached resume embeddings:", resume_emb.shape)
else:
    resume_emb = model.encode(
        cvs["Resume"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(emb_path, resume_emb)
    print("Saved resume embeddings:", resume_emb.shape)


Loaded cached resume embeddings: (962, 384)


In [15]:

# -------------------------
# Heuristic signals (taxonomy / seniority / skills)
# -------------------------

SKILLS = [
  # dev / data (giữ nguyên)
  "python","java","javascript","typescript","c#","c++","go","rust","php","ruby",
  "sql","mysql","postgresql","mongodb","redis",
  "react","angular","vue","node","next.js","nest","django","flask","fastapi","spring",
  "docker","kubernetes","aws","azure","gcp",
  "pandas","numpy","scikit-learn","tensorflow","pytorch","machine learning","deep learning",
  "tableau","power bi","spark","hadoop",

  # ✅ IT support / sysadmin (THÊM)
  "active directory","ad","windows","windows server","linux",
  "desktop support","helpdesk","help desk","service desk","it support",
  "technical support","troubleshooting","hardware","software installation",
  "office 365","microsoft 365","o365","outlook","exchange",
  "vpn","networking","dns","dhcp","tcp/ip","wifi",
  "jira","servicenow","ticketing","itil","remote support"
]


# build regex once (word boundaries; handle c++, c# specially)
_skill_patterns = []
for s in SKILLS:
    if s in ["c++","c#"]:
        pat = re.compile(rf"(?i)(?<!\w){re.escape(s)}(?!\w)")
    else:
        pat = re.compile(rf"(?i)\b{re.escape(s)}\b")
    _skill_patterns.append((s, pat))

def extract_skills(text: str):
    text = text or ""
    found = []
    for s, pat in _skill_patterns:
        if pat.search(text):
            found.append(s)
    return sorted(set(found))

# Taxonomy buckets
TAX_KEYWORDS = {
    "it_support": ["support","helpdesk","help desk","service desk","technician","desktop","field technician","troubleshooting","ticket","itil","incident"],
    "frontend": ["frontend","react","angular","vue","javascript","typescript","html","css","ui"],
    "backend":  ["backend","api","microservice","node","nestjs","spring","django","flask","fastapi","java","golang"],
    "data":     ["data","analytics","etl","sql","warehouse","bi","tableau","power bi","spark","hadoop","ml","machine learning"],
    "devops":   ["devops","sre","docker","kubernetes","ci/cd","aws","azure","gcp","terraform"],
    "security": ["security","soc","forensic","audit","iam","pentest"],
}


def infer_taxonomy(text: str) -> str:
    t = (text or "").lower()
    scores = {k:0 for k in TAX_KEYWORDS}
    for k, kws in TAX_KEYWORDS.items():
        for kw in kws:
            if kw in t:
                scores[k] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "other"

# Map resume Category -> taxonomy (heuristic)
def map_cv_category_to_tax(cat: str) -> str:
    c = (cat or "").lower()

    # ✅ IT support
    if any(x in c for x in ["support", "helpdesk", "service desk", "desktop", "technician", "system administrator", "network"]):
        return "it_support"

    if any(x in c for x in ["data", "analytics", "science", "ml", "ai"]):
        return "data"
    if any(x in c for x in ["devops", "cloud"]):
        return "devops"
    if any(x in c for x in ["security"]):
        return "security"
    if any(x in c for x in ["web", "frontend", "ui", "ux"]):
        return "frontend"
    if any(x in c for x in ["java", "backend", "python", "developer", "engineer"]):
        return "backend"
    return "other"


SEN_LEVELS = ["intern","junior","mid","senior","lead","manager"]
def infer_seniority(text: str) -> str:
    t = (text or "").lower()
    if any(x in t for x in ["intern", "internship", "trainee"]):
        return "intern"
    if any(x in t for x in ["junior", "jr", "entry level"]):
        return "junior"
    if any(x in t for x in ["senior", "sr", "principal", "staff"]):
        return "senior"
    if any(x in t for x in ["lead", "tech lead"]):
        return "lead"
    if any(x in t for x in ["manager", "head", "director"]):
        return "manager"
    return "mid"

def sen_mismatch(job_sen: str, cv_sen: str) -> int:
    # distance in ordered list; treat unknown as mid
    a = SEN_LEVELS.index(job_sen) if job_sen in SEN_LEVELS else SEN_LEVELS.index("mid")
    b = SEN_LEVELS.index(cv_sen)  if cv_sen  in SEN_LEVELS else SEN_LEVELS.index("mid")
    return abs(a-b)


- Bước 1 (Retrieval): mã hoá nội dung công việc và tính độ tương đồng embedding
  với toàn bộ CV để lấy ra Top-N ứng viên phù hợp nhất.
- Bước 2 (Feature extraction): với mỗi ứng viên, trích xuất ngành nghề,
  cấp độ kinh nghiệm và tập kỹ năng từ CV.
- Bước 3 (Reranking): tính điểm tổng hợp cho từng ứng viên bằng cách kết hợp
  độ tương đồng ngữ nghĩa, mức độ trùng khớp kỹ năng, độ phù hợp ngành nghề
  và mức độ lệch cấp độ kinh nghiệm.
- Bước 4 (Selection): sắp xếp theo điểm tổng hợp và chọn ra Top-K ứng viên cuối cùng.


In [16]:

# -------------------------
# Retrieval + Rerank
# -------------------------

def retrieve_topN(job_index: int, topN: int = 200):
    job_vec = model.encode([jobs.iloc[job_index]["job_text"]], normalize_embeddings=True)[0]
    sims = resume_emb @ job_vec
    idx = np.argsort(-sims)[:topN]
    return idx, sims[idx]

def rerank(job_index: int, topN: int = 200, topK: int = 10):
    job = jobs.iloc[job_index]
    job_text = job["job_text"]
    job_tax = infer_taxonomy(job_text)
    job_sen = infer_seniority(job["title"])
    job_skills = extract_skills(job_text)

    idx, sem_scores = retrieve_topN(job_index, topN)
    
    rows = []
    for cv_idx, sem in zip(idx, sem_scores):
        cv_text = cvs.iloc[cv_idx]["Resume"]
        cv_cat  = cvs.iloc[cv_idx]["Category"]
        cv_tax  = map_cv_category_to_tax(cv_cat)
        cv_sen  = infer_seniority(cv_text)

        cv_skills = extract_skills(cv_text)
        matched = sorted(set(job_skills).intersection(cv_skills))
        overlap = len(matched) / max(1, len(set(job_skills)))

        tax_match = 1.0 if (job_tax == cv_tax and job_tax != "other") else 0.0
        mismatch = sen_mismatch(job_sen, cv_sen)

        # composite score (tweak weights if you want)
        final = (
            0.70 * float(sem) +
            0.20 * float(overlap) +
            0.10 * float(tax_match) -
            0.05 * float(mismatch)
        )

        rows.append({
            "job_id": job.get("id", job_index),
            "job_title": job["title"],
            "cv_index": int(cv_idx),
            "cv_category": cv_cat,
            "semantic_score": float(sem),
            "final_score": float(final),
            "job_tax": job_tax,
            "cv_tax": cv_tax,
            "job_sen": job_sen,
            "cv_sen": cv_sen,
            "matched_skills": matched,
            "skill_overlap": float(overlap),
            "tax_match": float(tax_match),
            "seniority_mismatch": int(mismatch),
        })
    
    df = pd.DataFrame(rows).sort_values("final_score", ascending=False).head(topK).reset_index(drop=True)
    return df


# Demo run (change job index)
JOB_INDEX = 0
TOPN = 200
TOPK = 10

topk = rerank(JOB_INDEX, TOPN, TOPK)
topk



,job_id,job_title,cv_index,cv_category,semantic_score,final_score,job_tax,cv_tax,job_sen,cv_sen,matched_skills,skill_overlap,tax_match,seniority_mismatch
0,0,Support Technologist II 2024-02216,672,Network Security Engineer,0.469203,0.464156,it_support,it_support,mid,senior,"[hardware, ticketing, troubleshooting]",0.428571,1.0,1
1,0,Support Technologist II 2024-02216,677,Network Security Engineer,0.469203,0.464156,it_support,it_support,mid,senior,"[hardware, ticketing, troubleshooting]",0.428571,1.0,1
2,0,Support Technologist II 2024-02216,667,Network Security Engineer,0.469203,0.464156,it_support,it_support,mid,senior,"[hardware, ticketing, troubleshooting]",0.428571,1.0,1
3,0,Support Technologist II 2024-02216,662,Network Security Engineer,0.469203,0.464156,it_support,it_support,mid,senior,"[hardware, ticketing, troubleshooting]",0.428571,1.0,1
4,0,Support Technologist II 2024-02216,657,Network Security Engineer,0.469203,0.464156,it_support,it_support,mid,senior,"[hardware, ticketing, troubleshooting]",0.428571,1.0,1
5,0,Support Technologist II 2024-02216,673,Network Security Engineer,0.427232,0.434777,it_support,it_support,mid,senior,"[active directory, hardware, troubleshooting]",0.428571,1.0,1
6,0,Support Technologist II 2024-02216,658,Network Security Engineer,0.427232,0.434777,it_support,it_support,mid,senior,"[active directory, hardware, troubleshooting]",0.428571,1.0,1
7,0,Support Technologist II 2024-02216,678,Network Security Engineer,0.427232,0.434777,it_support,it_support,mid,senior,"[active directory, hardware, troubleshooting]",0.428571,1.0,1
8,0,Support Technologist II 2024-02216,663,Network Security Engineer,0.427232,0.434777,it_support,it_support,mid,senior,"[active directory, hardware, troubleshooting]",0.428571,1.0,1
9,0,Support Technologist II 2024-02216,668,Network Security Engineer,0.427232,0.434777,it_support,it_support,mid,senior,"[active directory, hardware, troubleshooting]",0.428571,1.0,1


In [20]:

# Export demo topK
out_path = OUT_DIR / "improved_topk.csv"
topk.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebooks/employer/EBD/outputs/candrec_emb/improved_topk.csv


In [21]:

# -------------------------
# Proxy Evaluation
# Evaluate across a sample of jobs: baseline vs improved
# -------------------------

def metrics_for_list(job_index: int, picked_cv_idx: np.ndarray, k: int):
    job = jobs.iloc[job_index]
    job_text = job["job_text"]
    job_tax = infer_taxonomy(job_text)
    job_sen = infer_seniority(job["title"])
    job_skills = extract_skills(job_text)

    cv_tax = [map_cv_category_to_tax(cvs.iloc[i]["Category"]) for i in picked_cv_idx[:k]]
    cv_sen = [infer_seniority(cvs.iloc[i]["Resume"]) for i in picked_cv_idx[:k]]
    cv_sk  = [extract_skills(cvs.iloc[i]["Resume"]) for i in picked_cv_idx[:k]]

    tax_hit = sum(1 for t in cv_tax if t == job_tax and job_tax != "other") / k
    sen_mis = sum(1 for s in cv_sen if sen_mismatch(job_sen, s) >= 2) / k

    overlaps = []
    expl = 0
    for skills in cv_sk:
        matched = set(job_skills).intersection(skills)
        overlaps.append(len(matched) / max(1, len(set(job_skills))))
        if (job_tax != "other") or (len(matched) > 0):
            expl += 1
    avg_overlap = float(np.mean(overlaps)) if overlaps else 0.0
    expl_rate = expl / k

    return tax_hit, sen_mis, avg_overlap, expl_rate

def baseline_rank(job_index: int, topK: int = 10):
    idx, sem = retrieve_topN(job_index, topN=topK)
    return idx

def improved_rank(job_index: int, topN: int = 200, topK: int = 10):
    df = rerank(job_index, topN=topN, topK=topK)
    return df["cv_index"].to_numpy()

# sample jobs
SAMPLE_JOBS = 100
K = 10
rng = np.random.default_rng(42)
job_indices = rng.choice(len(jobs), size=min(SAMPLE_JOBS, len(jobs)), replace=False)

rows = []
for ji in job_indices:
    base_idx = baseline_rank(int(ji), topK=K)
    imp_idx  = improved_rank(int(ji), topN=200, topK=K)

    b = metrics_for_list(int(ji), base_idx, K)
    i = metrics_for_list(int(ji), imp_idx,  K)

    rows.append({"type":"baseline","job_index":int(ji),
                 "TaxonomyHit@K":b[0],"SeniorityMismatch@K":b[1],
                 "AvgSkillOverlap@K":b[2],"ExplainabilityRate@K":b[3]})
    rows.append({"type":"improved","job_index":int(ji),
                 "TaxonomyHit@K":i[0],"SeniorityMismatch@K":i[1],
                 "AvgSkillOverlap@K":i[2],"ExplainabilityRate@K":i[3]})

metrics_rows = pd.DataFrame(rows)
summary = metrics_rows.groupby("type")[["TaxonomyHit@K","SeniorityMismatch@K","AvgSkillOverlap@K","ExplainabilityRate@K"]].mean()
summary


,TaxonomyHit@K,SeniorityMismatch@K,AvgSkillOverlap@K,ExplainabilityRate@K
type,,,,
baseline,0.162,0.632,0.160935,0.99
improved,0.431,0.156,0.418136,1.00


In [19]:

rows_path = OUT_DIR / "metrics_rows.csv"
sum_path  = OUT_DIR / "metrics_summary.csv"

metrics_rows.to_csv(rows_path, index=False)
summary.to_csv(sum_path)

print("Saved:", rows_path.resolve())
print("Saved:", sum_path.resolve())


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebooks/employer/EBD/outputs/candrec_emb/metrics_rows.csv
Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebooks/employer/EBD/outputs/candrec_emb/metrics_summary.csv
